<a href="https://colab.research.google.com/github/ArrushiTripathi2429/CutMix-Implementation/blob/main/CutMix_Experiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import torch

print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu130
GPU available: True
GPU: Tesla T4


In [10]:
from torchvision import datasets

train_dataset = datasets.CIFAR10(
    root="./data",
    train=True,
    download=True
)

test_dataset = datasets.CIFAR10(
    root="./data",
    train=False,
    download=True
)

print("Original training images:", len(train_dataset))
print("Original testing images:", len(test_dataset))

100%|██████████| 170M/170M [21:19<00:00, 133kB/s]


Original training images: 50000
Original testing images: 10000


In [11]:
print("Classes:")
print(train_dataset.classes)

print("\nNumber of classes:")
print(len(train_dataset.classes))

# Check one image
image, label = train_dataset[0]

print("\nImage shape:", image.size)
print("Label number:", label)
print("Label name:", train_dataset.classes[label])

Classes:
['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

Number of classes:
10

Image shape: (32, 32)
Label number: 6
Label name: frog


In [12]:
from torch.utils.data import Subset
import numpy as np

# Number of images per class
TRAIN_PER_CLASS = 300
TEST_PER_CLASS = 200

# Get labels
train_targets = np.array(train_dataset.targets)
test_targets = np.array(test_dataset.targets)

# Store selected indices
train_indices = []
test_indices = []

# Select images class by class
for class_id in range(10):

    # Indices belonging to this class
    train_class_indices = np.where(train_targets == class_id)[0]
    test_class_indices = np.where(test_targets == class_id)[0]

    # Shuffle indices
    np.random.shuffle(train_class_indices)
    np.random.shuffle(test_class_indices)

    # Take required number
    train_indices.extend(train_class_indices[:TRAIN_PER_CLASS])
    test_indices.extend(test_class_indices[:TEST_PER_CLASS])

# Create subsets
train_subset = Subset(train_dataset, train_indices)
test_subset = Subset(test_dataset, test_indices)

print("Training subset:", len(train_subset))
print("Testing subset:", len(test_subset))

Training subset: 3000
Testing subset: 2000


In [13]:
from collections import Counter

train_selected_labels = [
    train_dataset.targets[i] for i in train_indices
]

test_selected_labels = [
    test_dataset.targets[i] for i in test_indices
]

print("Training distribution:")
for class_id, count in sorted(Counter(train_selected_labels).items()):
    print(train_dataset.classes[class_id], ":", count)

print("\nTesting distribution:")
for class_id, count in sorted(Counter(test_selected_labels).items()):
    print(test_dataset.classes[class_id], ":", count)

Training distribution:
airplane : 300
automobile : 300
bird : 300
cat : 300
deer : 300
dog : 300
frog : 300
horse : 300
ship : 300
truck : 300

Testing distribution:
airplane : 200
automobile : 200
bird : 200
cat : 200
deer : 200
dog : 200
frog : 200
horse : 200
ship : 200
truck : 200


In [14]:
from torchvision import datasets, transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.4914, 0.4822, 0.4465),
        std=(0.2470, 0.2435, 0.2616)
    )
])

train_dataset = datasets.CIFAR10(
    root="./data",
    train=True,
    download=False,
    transform=transform
)

test_dataset = datasets.CIFAR10(
    root="./data",
    train=False,
    download=False,
    transform=transform
)

In [15]:
from torch.utils.data import Subset

train_subset = Subset(train_dataset, train_indices)
test_subset = Subset(test_dataset, test_indices)

print("Training subset:", len(train_subset))
print("Testing subset:", len(test_subset))

Training subset: 3000
Testing subset: 2000


In [16]:
from torch.utils.data import DataLoader

BATCH_SIZE = 64

train_loader = DataLoader(
    train_subset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

test_loader = DataLoader(
    test_subset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

In [17]:
images, labels = next(iter(train_loader))

print("Images shape:", images.shape)
print("Labels shape:", labels.shape)

Images shape: torch.Size([64, 3, 32, 32])
Labels shape: torch.Size([64])


In [18]:
import torch
import torch.nn as nn

In [19]:
class SimpleCNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [20]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = SimpleCNN().to(device)

print("Using device:", device)

Using device: cuda


In [21]:
images, labels = next(iter(train_loader))

images = images.to(device)
labels = labels.to(device)

outputs = model(images)

print("Input shape:", images.shape)
print("Output shape:", outputs.shape)

Input shape: torch.Size([64, 3, 32, 32])
Output shape: torch.Size([64, 10])


In [22]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [23]:
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU not available")

Device: cuda
GPU: Tesla T4


In [24]:
EPOCHS = 15

for epoch in range(EPOCHS):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        # Forward pass
        outputs = model(images)

        # Calculate loss
        loss = criterion(outputs, labels)

        # Remove old gradients
        optimizer.zero_grad()

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()

        # Statistics
        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(train_loader)
    accuracy = 100 * correct / total

    print(
        f"Epoch [{epoch+1}/{EPOCHS}] "
        f"Loss: {epoch_loss:.4f} "
        f"Accuracy: {accuracy:.2f}%"
    )

Epoch [1/15] Loss: 2.0020 Accuracy: 27.60%
Epoch [2/15] Loss: 1.6190 Accuracy: 42.50%
Epoch [3/15] Loss: 1.3714 Accuracy: 50.53%
Epoch [4/15] Loss: 1.2139 Accuracy: 56.77%
Epoch [5/15] Loss: 1.0814 Accuracy: 62.03%
Epoch [6/15] Loss: 0.9198 Accuracy: 67.53%
Epoch [7/15] Loss: 0.8025 Accuracy: 72.93%
Epoch [8/15] Loss: 0.6885 Accuracy: 76.43%
Epoch [9/15] Loss: 0.5591 Accuracy: 81.20%
Epoch [10/15] Loss: 0.4258 Accuracy: 86.30%
Epoch [11/15] Loss: 0.3074 Accuracy: 90.90%
Epoch [12/15] Loss: 0.2778 Accuracy: 91.63%
Epoch [13/15] Loss: 0.2084 Accuracy: 94.23%
Epoch [14/15] Loss: 0.1125 Accuracy: 97.70%
Epoch [15/15] Loss: 0.0796 Accuracy: 98.97%


In [25]:
model.eval()

correct = 0
total = 0
test_loss = 0.0

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        loss = criterion(outputs, labels)
        test_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

test_accuracy = 100 * correct / total
average_test_loss = test_loss / len(test_loader)

print(f"Test Loss: {average_test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.2f}%")

Test Loss: 2.3736
Test Accuracy: 51.95%
